In [ ]:
# 1.Импорт библиотек
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

# Настройка, чтобы графики отображались прямо в ноутбуке
%matplotlib inline

# 2.Загрузка данных
df = pd.read_csv('C:/Users/user/Desktop/CourseData/DA_TrainingSQLWeek3/Day11,12_SQL_S1_Katy.sql/S1_orders_dirty.csv')

# 3. Первичный осмотр
print('Размер данных:', df.shape)
display(df.head())
display(df.info())

In [ ]:
# 4. Очистка данных (Data Cleaning)

# 4.1. Конвертация дат из текста в формат datetime
df['order_date'] = pd.to_datetime(df['order_date'], format='mixed', errors='coerce')
df['ship_date'] = pd.to_datetime(df['ship_date'], format='mixed', errors='coerce')

# 4.2. Очистка и нормализация текста в колонке region (делаем первую букву заглавной)
df['region'] = df['region'].astype(str).str.lower().str.strip().str.title()

# 4.3. Исправляем опечатки в статусах
df['status'] = df['status'].replace({'Deliverd': 'Delivered', 'Canceled': 'Cancelled'})

# 4.4. Исправляем опечатки в способах оплаты
df['payment_method'] = df['payment_method'].replace({'Credit Crd': 'Credit Card'})

# 4.3. Обработка пропусков в payment_method. Заменяем пустые значения на ('Unknown')
df['payment_method'] = df['payment_method'].fillna('Unknown')

# 4.4. Удаляем полные дубли
df = df.drop_duplicates(keep='first').copy()

# 4.4. Проверка, что изменилось
display(df[['order_date', 'ship_date', 'region', 'payment_method']].head(20))
display(df.info())
print("Размер после удаления дублей:", df.shape)
print("--- РЕГИОНЫ ---")
print(df['region'].value_counts())

print("\n--- КАТЕГОРИИ ---")
print(df['category'].value_counts())

print("\n--- СТАТУСЫ ЗАКАЗОВ ---")
print(df['status'].value_counts())

print("\n--- СПОСОБЫ ОПЛАТЫ ---")
print(df['payment_method'].value_counts())

In [ ]:
# Остались проблемы в столбце  Region. Выводим все уникальные регионы с их длинной и скрытыми символами
for reg in df['region'].unique():
    print(f"'{reg}'| Длина: {len(reg)} | Скрытые символы: {repr(reg)}")

In [ ]:
# Скрытых символов не найдено. Поэтоиу ищем строки,где смешаны русские и английские буквы
import re

for reg in df['region'].unique():
    # Проверяем, есть ли в строке английские буквы
    has_latin = bool(re.search('[a-zA-Z]', reg))
    # Проверяем, есть ли русские
    has_cyrillic = bool(re.search('[а-яА-Я]', reg))
    if has_latin and has_cyrillic:
        print(f"НАЙДЕНА СМЕСЬ: {repr(reg)}")
        print(f"Коды символов: {[ord(c) for c in reg]}\n")

In [ ]:
# 1. Создаем таблицу перевода: латинские 'o' и 'a' -> кириллические 'о' и 'а'
# Это гарантированно разные буквы, так как мы используем их unicode-коды
trans_table = str.maketrans({'o': 'о', 'a': 'а', 'O': 'О', 'A': 'А'})

# 2. Применяем таблицу ко всей колонке
df['region'] = df['region'].str.translate(trans_table)

# 3. Возвращаем правильный регистр (Title Case), так как после замены 
# вторая часть слова 'петербург' могла остаться с маленькой буквы
df['region'] = df['region'].str.title()

# 4. Проверяем результат
print(df['region'].value_counts())

In [ ]:
# Проверка, после полной очистки данных
display(df[['order_date', 'ship_date', 'region', 'payment_method']].head())
display(df.info())
print("Размер после удаления дублей:", df.shape)
print("--- РЕГИОНЫ ---")
print(df['region'].value_counts())

print("\n--- КАТЕГОРИИ ---")
print(df['category'].value_counts())

print("\n--- СТАТУСЫ ЗАКАЗОВ ---")
print(df['status'].value_counts())

print("\n--- СПОСОБЫ ОПЛАТЫ ---")
print(df['payment_method'].value_counts())

In [ ]:
# 5. Базовые бизнес-метрики (Key Metrics)

# считаем общую выручку
total_revenue = df['total_amount'].sum()
# считаем количество уникальных заказов
total_orders = df['order_id'].nunique()
# считаем средний чек (AOV - Average Order Value)
aov = total_revenue / total_orders
# считаем количество уникальных клиентов
total_customers = df['customer_id'].nunique()

print(f'Общая выручка: {total_revenue:,.2f}руб.')
print(f'Всего заказов: {total_orders}')
print(f'Средний чек (AOV): {aov:,.2f}руб.')
print(f'Уникальных клиентов: {total_customers}')

In [ ]:
# 1. Проверка пропусков (NaT)
print("Пропуски в order_date:", df['order_date'].isna().sum())
print("Пропуски в ship_date:", df['ship_date'].isna().sum())

# 2. Проверка диапазона (от какой до какой даты у нас данные)
print(f"\nМин. дата заказа: {df['order_date'].min()}")
print(f"Макс. дата заказа: {df['order_date'].max()}")

# 3. Проверка бизнес-логики (дата доставки не может быть раньше даты заказа)
# Находим строки, где ship_date меньше order_date
invalid_dates = df[df['ship_date'] < df['order_date']]
print(f"\nКоличество заказов, где доставка была ДО заказа: {len(invalid_dates)}")

# Выводим первые 5 таких ошибок (если они есть), чтобы посмотреть
if len(invalid_dates) > 0:
    display(invalid_dates[['order_id', 'order_date', 'ship_date']].head())

In [ ]:
# Находим индексы (номера строк), где дата доставки раньше даты заказа
invalid_idx = df[df['ship_date'] < df['order_date']].index

# Заменяем неверные даты доставки на NaT (пустоту)
df.loc[invalid_idx, 'ship_date'] = pd.NaT

# Проверяем, что ошибок больше нет
print("Ошибок после чистки:", len(df[df['ship_date'] < df['order_date']]))

# Проверяем, сколько теперь пропусков в ship_date (было 96, станет 96 + 15 = 111)
print("Пропуски в ship_date после очистки:", df['ship_date'].isna().sum())

In [ ]:
# 6. Данные идеально очищены. Переходим к визуализации: Выручка по категориям

# группируем по категориям и суммируем выручку, сортируем по убыванию
category_revenue = df.groupby('category')['total_amount'].sum().sort_values(ascending=False)

# строим график
plt.figure(figsize=(10,5))
category_revenue.plot(kind='bar',color='teal')
plt.title('Общая выручка по категориям товаров')
plt.ylabel('Выручка (руб.)')
plt.xlabel('Категория')
plt.xticks(rotation=45)
plt.grid(axis='y', linestyle='--',alpha=0.7)
plt.tight_layout()

# Сохраняем график как картинку 
plt.savefig('revenue_by_category.png')
plt.show()

In [ ]:
# 7. Когортный анализ (Retention)

# Шаг 1: Определяем месяц первого заказа для каждого клиента (Когорта)
df['order_month'] = df['order_date'].dt.to_period('M')
first_orders = df.groupby('customer_id')['order_month'].min().rename('cohort_month')

# Шаг 2: Присоединяем когорту к каждому заказу клиента
df_cohort = df.merge(first_orders, on='customer_id', how='left')

# Шаг 3: Считаем номер месяца жизни (0 - первый месяц, 1 - второй и т.д.)
df_cohort['month_number'] = (df_cohort['order_month'] - df_cohort['cohort_month']).apply(lambda x: x.n)

# Шаг 4: Считаем количество уникальных клиентов в каждой когорте в каждом месяце
cohort_data = df_cohort.groupby(['cohort_month', 'month_number'])['customer_id'].nunique().reset_index()
cohort_data.columns = ['cohort_month', 'month_number', 'customer_count']

# Шаг 5: Превращаем в матрицу (строки - когорты, колонки - месяц жизни)
cohort_matrix = cohort_data.pivot(index='cohort_month', columns='month_number', values='customer_count')

# Шаг 6: Считаем Retention Rate (делим каждое значение на первый месяц когорты)
retention_matrix = cohort_matrix.div(cohort_matrix.iloc[:, 0], axis=0) * 100

# Выводим таблицу (округляем до 1 знака)
display(retention_matrix.round(1))

In [ ]:
import seaborn as sns

# Настройка размера графика
plt.figure(figsize=(15, 8))

# Строим тепловую карту
# cmap='Blues' - синяя цветовая гамма, annot=True - показывать цифры, fmt='.0f' - без десятых
sns.heatmap(retention_matrix, annot=True, fmt='.0f', cmap='Blues', linewidths=.5, cbar_kws={'label': 'Retention %'})

plt.title('Retention Rate по когортам (в %)')
plt.xlabel('Месяц жизни клиента')
plt.ylabel('Месяц первой покупки (Когорта)')

# Сохраняем для GitHub
plt.savefig('retention_heatmap.png')
plt.show()

In [ ]:
# 8. RFM-анализ (Сегментация клиентов)

# Берем максимальную дату в датасете + 1 день (как будто "сегодня")
snapshot_date = df['order_date'].max() + pd.Timedelta(days=1)

# Считаем R, F, M для каждого клиента
rfm = df.groupby('customer_id').agg({
    'order_date': lambda x: (snapshot_date - x.max()).days, # R: сколько дней прошло с последнего заказа
    'order_id': 'nunique',                                  # F: количество уникальных заказов
    'total_amount': 'sum'                                   # M: общая сумма покупок
}).reset_index()

rfm.columns = ['customer_id', 'Recency', 'Frequency', 'Monetary']

# Оцениваем каждого клиента от 1 до 5 (5 - лучший результат)
# Для R: чем МЕНЬШЕ дней, тем ЛУЧШЕ (поэтому инвертируем 5,4,3,2,1)
rfm['R_Score'] = pd.qcut(rfm['Recency'], 5, labels=[5, 4, 3, 2, 1])
rfm['F_Score'] = pd.qcut(rfm['Frequency'].rank(method='first'), 5, labels=[1, 2, 3, 4, 5])
rfm['M_Score'] = pd.qcut(rfm['Monetary'], 5, labels=[1, 2, 3, 4, 5])

# Переводим оценки в числа
rfm[['R_Score', 'F_Score', 'M_Score']] = rfm[['R_Score', 'F_Score', 'M_Score']].astype(int)

# Функция для распределения клиентов по бизнес-сегментам
def get_segment(row):
    if row['R_Score'] >= 4 and row['F_Score'] >= 4 and row['M_Score'] >= 4:
        return 'VIP'
    elif row['R_Score'] <= 2 and row['F_Score'] >= 4:
        return 'Под риском оттока'
    elif row['R_Score'] <= 2 and row['F_Score'] <= 2:
        return 'Спящие'
    elif row['R_Score'] >= 4 and row['F_Score'] == 1:
        return 'Новички'
    else:
        return 'Обычные'

# Применяем функцию ко всем строкам
rfm['Segment'] = rfm.apply(get_segment, axis=1)

# Выводим первые строки и статистику по сегментам
display(rfm.head())
print("\nКоличество клиентов в каждом сегменте:")
print(rfm['Segment'].value_counts())

In [ ]:
# 9. Визуализация RFM-сегментов

# Считаем количество клиентов в каждом сегменте
segment_counts = rfm['Segment'].value_counts()

# Строим круговую диаграмму
plt.figure(figsize=(8, 8))
# Цвета для сегментов
colors = ['#4C72B0', '#DD8452', '#55A868', '#C44E52', '#8172B3'] 

plt.pie(segment_counts, 
        labels=segment_counts.index, 
        autopct='%1.1f%%', # показываем проценты
        startangle=140, 
        colors=colors,
        wedgeprops={'edgecolor': 'white'}) # белая обводка для красоты

plt.title('Распределение клиентов по RFM-сегментам', fontsize=14)
plt.tight_layout()

# Сохраняем график для GitHub
plt.savefig('rfm_segments.png')
plt.show()
